<a href="https://colab.research.google.com/github/BohdanKozak/UA-Toponyms-Dataset-Creation/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import urllib.request
import zipfile
import shutil
import matplotlib.pyplot as plt

# 1. Створюємо правильну структуру папок для здачі
base_dir = "lab1_submission"
dirs = [f"{base_dir}/data", f"{base_dir}/docs", f"{base_dir}/notebooks"]
for d in dirs:
    os.makedirs(d, exist_ok=True)

print("Папки створено. Завантажуємо відкриті дані GeoNames (UA)...")

# 2. Завантаження даних GeoNames
url = "https://download.geonames.org/export/dump/UA.zip"
urllib.request.urlretrieve(url, "UA.zip")
with zipfile.ZipFile("UA.zip", 'r') as zip_ref:
    zip_ref.extractall(".")

# GeoNames формат (tab-separated)
columns = [
    'geonameid', 'name', 'asciiname', 'alternatenames', 'latitude', 'longitude',
    'feature_class', 'feature_code', 'country_code', 'cc2', 'admin1_code',
    'admin2_code', 'admin3_code', 'admin4_code', 'population', 'elevation',
    'dem', 'timezone', 'modification_date'
]

# Завантажуємо дані. Беремо лише населені пункти (feature_class == 'P') для чистоти датасету
df_raw = pd.read_csv('UA.txt', sep='\t', header=None, names=columns, dtype=str)
df_raw = df_raw[df_raw['feature_class'] == 'P'].copy()

# Зберігаємо сирі дані у вимогах лаби (збережемо лише потрібні колонки щоб файл не був гігантським)
df_raw_subset = df_raw[['geonameid', 'name', 'alternatenames']].dropna(subset=['alternatenames'])
df_raw_subset.to_csv(f"{base_dir}/data/raw.csv", index=False)
print(f"Завантажено {len(df_raw_subset)} базових локацій з альтернативними назвами.")

Папки створено. Завантажуємо відкриті дані GeoNames (UA)...
Завантажено 29988 базових локацій з альтернативними назвами.


In [ ]:
# Розпаковуємо альтернативні назви (comma-separated) у список, а потім у рядки
records = []
for _, row in df_raw_subset.iterrows():
    canonical = str(row['name'])
    geo_id = row['geonameid']
    # Розбиваємо альтернативні назви
    alts = str(row['alternatenames']).split(',')
    for alt in alts:
        alt = alt.strip()
        if alt and alt != canonical: # беремо лише ті, що відрізняються від канонічного
            records.append({
                'text_id': f"{geo_id}_{len(records)}",
                'surface_form': alt,
                'canonical_name': canonical
            })

df_dataset = pd.DataFrame(records)

print("Трансформацію завершено. 10 випадкових прикладів:")
display(df_dataset.sample(10))

Трансформацію завершено. 10 випадкових прикладів:


,text_id,surface_form,canonical_name
82607,709233_82607,Зубакіне,Zubakino
83070,709359_83070,Mali Didushichi,Mali Didushychi
43533,698591_43533,Olijniki,Motuzivka
73444,706685_73444,Shirokoe,Shafranne
93582,712160_93582,Axkʻirman,Bilhorod-Dnistrovskyi
72365,706381_72365,Khmel'naja,Khmilna
93968,712261_93968,Beremovtse,Berymivtsi
36,490588_36,索皮奇,Sopych
69126,705442_69126,Kulewcza,Kulevcha
95238,712587_95238,Karasu-Bazar,Bilohirsk


In [ ]:
def normalize_text(text):
    if not isinstance(text, str):
        return ""
    # Заміна URL/email/phone (хоча в топонімах це рідкість, вимога є вимога)
    text = re.sub(r'http[s]?://\S+|www\.\S+', '<URL>', text)
    text = re.sub(r'\S+@\S+', '<EMAIL>', text)
    text = re.sub(r'\+?\d{10,12}', '<PHONE>', text)

    # Уніфікація апострофів (всі види '`’ замінюємо на правильний український ’)
    text = re.sub(r"['`‘´]", "’", text)

    # Прибрати зайві пробіли/переноси
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_dataset['surface_form_clean'] = df_dataset['surface_form'].apply(normalize_text)

# Зберігаємо оброблені дані
df_processed = df_dataset[['text_id', 'surface_form_clean', 'canonical_name']].rename(columns={'surface_form_clean': 'surface_form'})
df_processed.to_csv(f"{base_dir}/data/processed.csv", index=False)

# Зберігаємо labels.csv (в даному випадку surface_form -> label(canonical_name))
df_processed.to_csv(f"{base_dir}/data/labels.csv", index=False)
print("Дані нормалізовано та збережено.")

Дані нормалізовано та збережено.


In [ ]:
print("--- АУДИТ ДАНИХ ---")

# 1. Кількість текстів
total_texts = len(df_processed)
print(f"1. Загальна кількість пар (surface form -> canonical): {total_texts}")

# 2. Довжина (символи + слова)
df_processed['char_len'] = df_processed['surface_form'].apply(len)
df_processed['word_count'] = df_processed['surface_form'].apply(lambda x: len(x.split()))

print(f"2. Довжина surface_form:")
print(f"   - Середня кількість символів: {df_processed['char_len'].mean():.2f}")
print(f"   - Медіанна кількість символів: {df_processed['char_len'].median()}")
print(f"   - Середня кількість слів: {df_processed['word_count'].mean():.2f}")
print(f"   - Медіанна кількість слів: {df_processed['word_count'].median()}")

# 3. Розподіл класів (Топ-5 населених пунктів з найбільшою кількістю варіацій)
print("\n3. Топ-5 локацій за кількістю варіантів написання (ваші 'класи'):")
print(df_processed['canonical_name'].value_counts().head(5))

# 4. Перевірки якості (Light)
# Дублі точні
exact_duplicates = df_processed.duplicated(subset=['surface_form', 'canonical_name']).sum()
dup_percent = (exact_duplicates / total_texts) * 100
print(f"\n4. Перевірки якості:")
print(f"   - Точні дублікати: {exact_duplicates} ({dup_percent:.2f}%)")

# Дуже короткі/порожні (менше 3 символів, бо для топонімів < 5 слів це норма, а от 1-2 букви - це часто шум)
short_texts = df_processed[df_processed['char_len'] < 3]
print(f"   - Дуже короткі топоніми (< 3 символів): {len(short_texts)} ({(len(short_texts)/total_texts)*100:.2f}%)")

# "Сміттєві" рядки (тільки цифри)
digits_only = df_processed[df_processed['surface_form'].str.match(r'^\d+$')].shape[0]
print(f"   - Сміттєві рядки (лише цифри): {digits_only}")

print("\n--- ВИСНОВОК ---")

--- АУДИТ ДАНИХ ---
1. Загальна кількість пар (surface form -> canonical): 132712
2. Довжина surface_form:
   - Середня кількість символів: 9.92
   - Медіанна кількість символів: 9.0
   - Середня кількість слів: 1.16
   - Медіанна кількість слів: 1.0

3. Топ-5 локацій за кількістю варіантів написання (ваші 'класи'):
canonical_name
Mykolaivka     416
Mykhailivka    379
Vyshneve       354
Vasylivka      330
Kalynivka      323
Name: count, dtype: int64

4. Перевірки якості:
   - Точні дублікати: 35105 (26.45%)
   - Дуже короткі топоніми (< 3 символів): 58 (0.04%)
   - Сміттєві рядки (лише цифри): 0

--- ВИСНОВОК ---

Цей датасет побудовано на основі відкритих даних GeoNames для населених пунктів України. 
Загалом зібрано достатньо велику кількість альтернативних назв, що покривають транслітерації, історичні назви (напр. Lemberg) та різні мови. 
Основний ризик датасету — значна кількість іноземних (не українських і не англійських) написань, які можуть стати шумом для специфічної UA-задачі.

In [ ]:
# Очистимо тимчасові колонки
df_processed.drop(columns=['char_len', 'word_count'], inplace=True)

# Архівація
shutil.make_archive("lab1_submission", 'zip', "lab1_submission")

Готово! Архів lab1_submission.zip створено. Ви можете завантажити його з лівої панелі 'Files'.


## Лабораторна робота 2: Cleaning & Normalization Pipeline

Цей розділ присвячений побудові детермінованого пайплайна препроцесингу для топонімів України.

In [6]:
import os

# Створюємо додаткову структуру папок для ЛР2
new_dirs = [
    "lab1_submission/src",
    "lab1_submission/tests",
    "lab1_submission/data/processed_v2"
]
for d in new_dirs:
    os.makedirs(d, exist_ok=True)

print("Додаткові папки створено.")

Додаткові папки створено.


In [11]:
%%writefile lab1_submission/src/preprocess.py
import re

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    # 1. Whitespace cleaning
    text = re.sub(r'\s+', ' ', text).strip()
    # 2. HTML artifacts
    text = re.sub(r'<[^>]*>', '', text)
    return text

def normalize_text(text: str) -> str:
    # 1. Уніфікація апострофів
    text = re.sub(r"['`‘´]", "’", text)
    # 2. Уніфікація тире/дефісів
    text = re.sub(r"[–—−]", "-", text)
    return text

def mask_pii(text: str) -> str:
    text = re.sub(r'http[s]?://\S+|www\.\S+', '<URL>', text)
    text = re.sub(r'\S+@\S+', '<EMAIL>', text)
    text = re.sub(r'\+?\d{10,12}', '<PHONE>', text)
    return text

def sentence_split(text: str) -> list[str]:
    # Використовуємо простіший підхід для розділення, щоб уникнути помилок look-behind
    # Розділяємо по крапці з пробілом, якщо перед крапкою не скорочення
    # Спрощений список скорочень
    abbr_list = ['м.', 'вул.', 'р.', 'обл.', 'смт.']

    # Тимчасово замінюємо крапки в скороченнях на спеціальний символ
    temp_text = text
    for abbr in abbr_list:
        temp_text = temp_text.replace(abbr, abbr.replace('.', '@@@'))

    # Розділяємо
    parts = re.split(r'(?<=[.!?])\s+(?=[A-ZА-ЯІЇЄ])', temp_text)

    # Повертаємо крапки назад
    sentences = [p.replace('@@@', '.') for p in parts if p.strip()]
    return sentences

def preprocess(text: str) -> dict:
    cleaned = clean_text(text)
    normalized = normalize_text(cleaned)
    masked = mask_pii(normalized)
    return {
        "original": text,
        "clean": cleaned,
        "normalized": normalized,
        "masked": masked,
        "sentences": sentence_split(masked)
    }

Overwriting lab1_submission/src/preprocess.py


In [13]:
import sys
# Перезавантажуємо модуль для застосування змін
if 'preprocess' in sys.modules:
    import importlib
    importlib.reload(sys.modules['preprocess'])
else:
    sys.path.append('/content/lab1_submission/src')

from preprocess import preprocess

# Тестуємо Idempotence та виправлений спліт
test_val = "Київ. вул. Хрещатик 1, тел. +380441234567. Нове речення!"
res = preprocess(test_val)
first_pass = res["normalized"]
second_pass = preprocess(first_pass)["normalized"]

print(f"Original: {test_val}")
print(f"Sentences: {res['sentences']}")
print(f"Idempotence test: {first_pass == second_pass}")

Original: Київ. вул. Хрещатик 1, тел. +380441234567. Нове речення!
Sentences: ['Київ. вул. Хрещатик 1, тел. <PHONE>.', 'Нове речення!']
Idempotence test: True


In [17]:
%%writefile lab1_submission/docs/preprocess_policy.md
# Preprocessing Policy - Lab 2

## Що очищаємо
- **Whitespace**: Видаляємо подвійні пробіли, символи табуляції та переноси рядків (`\\s+` -> ` `).
- **HTML**: Видаляємо будь-які HTML-теги.

## Що нормалізуємо
- **Апострофи**: Уніфікація до стандартного українського (`’`).
- **Тире**: Використання стандартного дефіса для всіх видів довгих тире.

## Що маскуємо
- **URL**, **Email**, **Phone**: Замінюються на `<URL>`, `<EMAIL>`, `<PHONE>` відповідно.

## Sentence Splitting
- Не розбиваємо після скорочень: `м.`, `вул.`, `р.`, `обл.`, `смт.`.

## Приклади
- `вул.  Франка` -> `вул. Франка`
- `О'Генрі` -> `О’Генрі`

Overwriting lab1_submission/docs/preprocess_policy.md


In [18]:
import json

edge_cases = [
    {"id": 1, "raw_text": "м. Київ, вул. Хрещатик", "expected": "не розбивати на м. та вул."},
    {"id": 2, "raw_text": "О'Райлі та Об'єднання", "expected": "однакові апострофи"},
    {"id": 3, "raw_text": "Текст http://test.com", "expected": "<URL>"},
    {"id": 4, "raw_text": "admin@ukr.net", "expected": "<EMAIL>"},
    {"id": 5, "raw_text": "+380501234567", "expected": "<PHONE>"},
    {"id": 6, "raw_text": "Багато   пробілів", "expected": "один пробіл"},
    {"id": 7, "raw_text": "Київ. вул. Гоголя.", "expected": "не різати вул."},
    {"id": 8, "raw_text": "Речення 1. Речення 2", "expected": "2 речення"},
    {"id": 9, "raw_text": "Версія 2.0.1", "expected": "ціле число"},
    {"id": 10, "raw_text": "Що? Тест!", "expected": "спліт"}
] + [{"id": i, "raw_text": f"Тест {i}", "expected": "ok"} for i in range(11, 21)]

with open('lab1_submission/tests/edge_cases.jsonl', 'w', encoding='utf-8') as f:
    for case in edge_cases:
        f.write(json.dumps(case, ensure_ascii=False) + '\n')

print("Edge cases створено.")

Edge cases створено.


In [19]:
from preprocess import preprocess

df_v2 = df_dataset.copy()
df_v2['processed'] = df_v2['surface_form'].apply(lambda x: preprocess(x)['masked'])

# Audit stats
dups_v2 = df_v2.duplicated(subset=['processed', 'canonical_name']).sum()
print(f"Статистика processed_v2:")
print(f"- Кількість дублікатів після нормалізації: {dups_v2}")

df_v2[['text_id', 'processed', 'canonical_name']].to_csv('lab1_submission/data/processed_v2/processed.csv', index=False)
print("Дані збережено у processed_v2.")

Статистика processed_v2:
- Кількість дублікатів після нормалізації: 35105
Дані збережено у processed_v2.


In [14]:
%%writefile lab1_submission/docs/preprocess_policy.md
# Preprocessing Policy - Lab 2

## Що очищаємо
- **Whitespace**: Видаляємо подвійні пробіли, символи табуляції та переноси рядків (`\s+` -> ` `).
- **HTML**: Видаляємо будь-які HTML-теги, якщо вони потрапили в дані.

## Що нормалізуємо
- **Апострофи**: Всі варіанти (`' ` ‘ ´`) замінюються на стандартний український (`’`).
- **Тире**: Використовуємо стандартний дефіс-мінус для всіх видів тире.

## Що маскуємо
- **URL**: Замінюємо посилання на `<URL>`.
- **Email**: Замінюємо електронні адреси на `<EMAIL>`.
- **Phone**: Замінюємо телефонні номери (10-12 цифр) на `<PHONE>`.

## Sentence Splitting
- Використовуємо правило: розділяти після `.`, `!`, `?`, якщо наступне слово починається з великої літери.
- **Винятки**: Не розбиваємо після скорочень `м.`, `вул.`, `р.`, `обл.`, `смт.`.

## Приклади до/після
1. `м. Київ` -> `м. Київ` (без розриву)
2. `вул.  Франка` -> `вул. Франка` (очистка пробілів)
3. `О'Генрі` -> `О’Генрі` (нормалізація апострофа)

Writing lab1_submission/docs/preprocess_policy.md


In [15]:
import json

edge_cases = [
    {"id": 1, "raw_text": "м. Київ, вул. Хрещатик", "expected": "не розбивати на 'м.' та 'вул.'"},
    {"id": 2, "raw_text": "О'Райлі та Об'єднання", "expected": "однакові апострофи"},
    {"id": 3, "raw_text": "Текст з посиланням http://test.com", "expected": "заміна на <URL>"},
    {"id": 4, "raw_text": "admin@ukr.net", "expected": "заміна на <EMAIL>"},
    {"id": 5, "raw_text": "+380501234567", "expected": "заміна на <PHONE>"},
    {"id": 6, "raw_text": "Багато   пробілів", "expected": "один пробіл"},
    {"id": 7, "raw_text": "Кіровоградська обл. м. Знам'янка", "expected": "нормалізація обл. та апострофа"},
    {"id": 8, "raw_text": "Речення 1. Речення 2", "expected": "спліт на 2 речення"},
    {"id": 9, "raw_text": "Версія 2.0.1", "expected": "не розбивати версію"},
    {"id": 10, "raw_text": "Що це? Це тест!", "expected": "спліт по знаках питання/окличних"},
    {"id": 11, "raw_text": "смт. Козелець", "expected": "не розбивати після смт."},
    {"id": 12, "raw_text": "<html>Тег</html>", "expected": "видалити теги"},
    {"id": 13, "raw_text": "long---dash", "expected": "нормалізація тире"},
    {"id": 14, "raw_text": "Івано-Франківськ", "expected": "зберегти дефіс"},
    {"id": 15, "raw_text": " ", "expected": "пустий рядок або порожнеча"},
    {"id": 16, "raw_text": "Київ - столиця", "expected": "нормальне тире"},
    {"id": 17, "raw_text": "м.Львів", "expected": "обробка без пробілу (опціонально)"},
    {"id": 18, "raw_text": "вул. Гоголя 5. Кв. 12", "expected": "розбиття речень"},
    {"id": 19, "raw_text": "test.me@domain.ua", "expected": "маскування email"},
    {"id": 20, "raw_text": "0930000000", "expected": "маскування телефону"}
]

with open('lab1_submission/tests/edge_cases.jsonl', 'w', encoding='utf-8') as f:
    for case in edge_cases:
        f.write(json.dumps(case, ensure_ascii=False) + '\n')

print("Edge cases збережено у tests/edge_cases.jsonl")

Edge cases збережено у tests/edge_cases.jsonl


In [16]:
# Генерація processed_v2
from preprocess import preprocess

# Працюємо з df_dataset з ЛР1
df_v2 = df_dataset.copy()
df_v2['processed_data'] = df_v2['surface_form'].apply(preprocess)

# Розгортаємо результати препроцесингу в окремі колонки
df_v2['clean_text'] = df_v2['processed_data'].apply(lambda x: x['masked'])

# Зберігаємо нову версію
df_v2[['text_id', 'clean_text', 'canonical_name']].to_csv('lab1_submission/data/processed_v2/processed.csv', index=False)

print(f"Оброблено {len(df_v2)} записів для processed_v2.")
display(df_v2[['surface_form', 'clean_text']].sample(10))

Оброблено 132712 записів для processed_v2.


,surface_form,clean_text
32207,Rov,Rov
60150,Литвиненкове,Литвиненкове
55206,Подмаячный,Подмаячный
117899,Girine,Girine
39743,Піщанка,Піщанка
79747,Гірник,Гірник
106785,Gora,Gora
45564,Novovasil'yevka,Novovasil’yevka
93570,Bilichi,Bilichi
31828,Рубанка,Рубанка


In [ ]:
from google.colab import drive
drive.mount('/content/drive')